In [8]:
import os
import json
import glob
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

In [9]:
BASEDIR = '/Users/egyptdj/github/bruit-classification'
DATADIR = os.path.join(BASEDIR, 'data_v5')
MODELDIR = os.path.join(BASEDIR, 'model')

MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
SAMPLE_RATE = 16000
BATCH_SIZE = 1

In [10]:
train_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'train', '**/*.wav')), dtype='object')
train_audio_label_list = [f.split('/')[-2] for f in train_audio_file_list]

In [11]:
val_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'val', '**/*.wav')), dtype='object')
val_audio_label_list = [f.split('/')[-2] for f in val_audio_file_list]

In [12]:
test_audio_file_list = pd.Series(glob.glob(os.path.join(DATADIR, 'test', '**/*.wav')), dtype='object')
test_audio_label_list = [f.split('/')[-2] for f in test_audio_file_list]

In [13]:
class_name_dict = {}
i = 0
for label in train_audio_label_list:
  if not label in list(class_name_dict.keys()):
    class_name_dict[label] = i
    i += 1

In [14]:
with open(os.path.join(BASEDIR, 'class_name_dict.json'), 'w') as f:
  json.dump(class_name_dict, f)

In [15]:
train_audio_label_list = pd.Series([class_name_dict[i] for i in train_audio_label_list], dtype='int64')
val_audio_label_list = pd.Series([class_name_dict[i] for i in val_audio_label_list], dtype='int64')
test_audio_label_list = pd.Series([class_name_dict[i] for i in test_audio_label_list], dtype='int64')

In [16]:
train_audio_label_list_step1 = train_audio_label_list.map(lambda x: 0 if x==class_name_dict['background'] else 1)
val_audio_label_list_step1 = val_audio_label_list.map(lambda x: 0 if x==class_name_dict['background'] else 1)
test_audio_label_list_step1 = test_audio_label_list.map(lambda x: 0 if x==class_name_dict['background'] else 1)

In [17]:
train_audio_label_list_step2 = train_audio_label_list.replace({class_name_dict['background']: -1, class_name_dict['normal']: 0, class_name_dict['bruit']: 1, class_name_dict['normal-noise']: 2, class_name_dict['bruit-noise']: 2})
val_audio_label_list_step2 = val_audio_label_list.replace({class_name_dict['background']: -1, class_name_dict['normal']: 0, class_name_dict['bruit']: 1, class_name_dict['normal-noise']: 2, class_name_dict['bruit-noise']: 2})
test_audio_label_list_step2 = test_audio_label_list.replace({class_name_dict['background']: -1, class_name_dict['normal']: 0, class_name_dict['bruit']: 1, class_name_dict['normal-noise']: 2, class_name_dict['bruit-noise']: 2})

In [18]:
train_ds1 = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list_step1))
val_ds1 = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list_step1))
test_ds1 = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list_step1))

train_ds2 = tf.data.Dataset.from_tensor_slices((train_audio_file_list, train_audio_label_list_step2)).filter(lambda x, y: tf.math.logical_not(tf.math.equal(y, -1)))
val_ds2 = tf.data.Dataset.from_tensor_slices((val_audio_file_list, val_audio_label_list_step2)).filter(lambda x, y: tf.math.logical_not(tf.math.equal(y, -1)))
test_ds2 = tf.data.Dataset.from_tensor_slices((test_audio_file_list, test_audio_label_list_step2)).filter(lambda x, y: tf.math.logical_not(tf.math.equal(y, -1)))

In [19]:
# *.wav 파일을 경로에서부터 오디오파일로 읽어오기

@tf.function
def load_wav_and_label(audio_file, label):
  audio = tf.io.read_file(audio_file)
  audio, sample_rate = tf.audio.decode_wav(audio)
  audio = tf.squeeze(audio, axis=-1)
  return audio, label

In [20]:
yamnet_model = hub.load(MODEL_HANDLE)

2023-07-04 16:14:07.188167: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_3' with dtype int32 and shape [?]
	 [[{{node inputs_3}}]]
2023-07-04 16:14:07.189297: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [3]
	 [[{{node inputs_1}}]]
2023-07-04 16:14:07.189346: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [3]
	 [[{{node inputs_1}}]]
2023-07-04 16:14

In [21]:
@tf.function
def extract_embedding(wav_data, label):
  scores, embeddings, spectrogram = yamnet_model(wav_data)
  num_embeddings = tf.shape(embeddings)[0]
  return embeddings, tf.repeat(label, num_embeddings)

In [22]:
model_step1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    tf.keras.layers.Dense(2)
])

model_step2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024), dtype=tf.float32, name='yamnet_embedding'),
    tf.keras.layers.Dense(3)
])

In [23]:
model_step1.summary()
model_step2.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 2)                 2050      
                                                                 
Total params: 2,050
Trainable params: 2,050
Non-trainable params: 0
_________________________________________________________________
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 3)                 3075      
                                                                 
Total params: 3,075
Trainable params: 3,075
Non-trainable params: 0
_________________________________________________________________


In [24]:
model_step1.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-5,
              ),
              metrics=['accuracy'],
)

In [25]:
model_step2.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(
                  learning_rate=1e-5,
              ),
              metrics=['accuracy'],
)

In [26]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

In [27]:
train_ds1 = train_ds1.map(load_wav_and_label).map(extract_embedding).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds1 = val_ds1.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds1 = test_ds1.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds2 = train_ds2.map(load_wav_and_label).map(extract_embedding).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds2 = val_ds2.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds2 = test_ds2.map(load_wav_and_label).map(extract_embedding).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

2023-07-04 16:14:08.706412: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'waveform' with dtype float and shape [?]
	 [[{{node waveform}}]]
2023-07-04 16:14:08.729149: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'wav_data' with dtype float and shape [?]
	 [[{{node wav_data}}]]
2023-07-04 16:14:08.756557: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'args_0' with dtype float and shape [?]
	 [[{{node args_0}}]]
2023-07-04 16:14:08.

In [28]:
history1 = model_step1.fit(train_ds1,
                    epochs=100,
                    validation_data=val_ds1,
                    callbacks=callback
                    )

2023-07-04 16:14:09.042639: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_33' with dtype resource
	 [[{{node Placeholder/_33}}]]
2023-07-04 16:14:09.044462: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_137' with dtype resource
	 [[{{node Placeholder/_137}}]]


Epoch 1/100


2023-07-04 16:14:09.176894: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2023-07-04 16:14:19.227148: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 302 of 1000
2023-07-04 16:14:29.228435: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 641 of 1000
2023-07-04 16:14:39.258622: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 893 of 1000


513/985 [==============>...............] - ETA: 0s - loss: 0.5588 - accuracy: 0.8115 

2023-07-04 16:14:43.276762: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


871/985 [=========================>....] - ETA: 0s - loss: 0.5523 - accuracy: 0.8102

2023-07-04 16:14:43.569798: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_12' with dtype resource
	 [[{{node Placeholder/_12}}]]


985/985 [==============================] - 37s 3ms/step - loss: 0.5482 - accuracy: 0.8191 - val_loss: 0.4553 - val_accuracy: 0.8585
Epoch 2/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4938 - accuracy: 0.8287 - val_loss: 0.4155 - val_accuracy: 0.8585
Epoch 3/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4602 - accuracy: 0.8282 - val_loss: 0.3900 - val_accuracy: 0.8585
Epoch 4/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4380 - accuracy: 0.8280 - val_loss: 0.3733 - val_accuracy: 0.8571
Epoch 5/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4220 - accuracy: 0.8298 - val_loss: 0.3613 - val_accuracy: 0.8598
Epoch 6/100
985/985 [==============================] - 3s 3ms/step - loss: 0.4096 - accuracy: 0.8318 - val_loss: 0.3523 - val_accuracy: 0.8611
Epoch 7/100
985/985 [==============================] - 3s 3ms/step - loss: 0.3995 - accuracy: 0.8353 - val_loss: 0.3455 - val_accuracy: 0.8677
Epoch 8/10

In [29]:
model_step1.evaluate(test_ds1)

2023-07-04 16:19:17.801029: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_50' with dtype float and shape [256]
	 [[{{node Placeholder/_50}}]]


68/68 [==============================] - 3s 37ms/step - loss: 0.2010 - accuracy: 0.8373


[0.20103387534618378, 0.8372591137886047]

In [30]:
history2 = model_step2.fit(train_ds2,
                    epochs=100,
                    validation_data=val_ds2,
                    callbacks=callback
                    )

Epoch 1/100


2023-07-04 16:19:20.836951: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_41' with dtype float and shape [128]
	 [[{{node Placeholder/_41}}]]
2023-07-04 16:19:20.838703: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_54' with dtype resource
	 [[{{node Placeholder/_54}}]]
2023-07-04 16:19:30.988032: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 325 of 1000
2023-07-04 16:19:40.981896: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 611 of 1000


    479/Unknown - 28s 315us/step - loss: 1.2090 - accuracy: 0.2249

2023-07-04 16:19:48.438058: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


    648/Unknown - 28s 310us/step - loss: 1.2011 - accuracy: 0.2379

2023-07-04 16:19:48.859358: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_123' with dtype resource
	 [[{{node Placeholder/_123}}]]


785/785 [==============================] - 30s 3ms/step - loss: 1.1997 - accuracy: 0.2409 - val_loss: 1.1100 - val_accuracy: 0.3628
Epoch 2/100
785/785 [==============================] - 2s 3ms/step - loss: 1.1493 - accuracy: 0.2952 - val_loss: 1.0601 - val_accuracy: 0.4713
Epoch 3/100
785/785 [==============================] - 2s 3ms/step - loss: 1.1082 - accuracy: 0.3611 - val_loss: 1.0225 - val_accuracy: 0.5271
Epoch 4/100
785/785 [==============================] - 2s 3ms/step - loss: 1.0751 - accuracy: 0.4123 - val_loss: 0.9919 - val_accuracy: 0.5504
Epoch 5/100
785/785 [==============================] - 2s 3ms/step - loss: 1.0470 - accuracy: 0.4492 - val_loss: 0.9675 - val_accuracy: 0.5721
Epoch 6/100
785/785 [==============================] - 2s 3ms/step - loss: 1.0231 - accuracy: 0.4801 - val_loss: 0.9480 - val_accuracy: 0.5922
Epoch 7/100
785/785 [==============================] - 2s 3ms/step - loss: 1.0020 - accuracy: 0.5045 - val_loss: 0.9318 - val_accuracy: 0.6016
Epoch 8/10

In [31]:
model_step2.evaluate(test_ds2)

2023-07-04 16:23:44.007196: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_118' with dtype resource
	 [[{{node Placeholder/_118}}]]


58/58 [==============================] - 2s 35ms/step - loss: 0.7241 - accuracy: 0.7362


[0.7241054773330688, 0.736248254776001]

In [34]:
input_segment = tf.keras.layers.Input(shape=(), dtype=tf.float32, name='audio')
embedding_extraction_layer = hub.KerasLayer(MODEL_HANDLE,
                                            trainable=False, name='yamnet')
_, embeddings, _ = embedding_extraction_layer(input_segment)
step1_output = tf.cast(tf.argmax(tf.math.reduce_mean(model_step1(embeddings), axis=0), axis=0), tf.float32)
step2_output = tf.math.reduce_mean(model_step2(embeddings), axis=0)
final_output = tf.multiply(step1_output, step2_output)

2023-07-04 16:23:46.526859: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_3' with dtype int32 and shape [?]
	 [[{{node inputs_3}}]]
2023-07-04 16:23:46.527787: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [3]
	 [[{{node inputs_1}}]]
2023-07-04 16:23:46.527834: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [3]
	 [[{{node inputs_1}}]]
2023-07-04 16:23

In [35]:
serving_model = tf.keras.Model(input_segment, final_output)

In [36]:
serving_model.save(MODELDIR, include_optimizer=False)

2023-07-04 16:23:48.069042: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'waveform' with dtype float and shape [?]
	 [[{{node waveform}}]]
2023-07-04 16:23:48.095954: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'audio' with dtype float and shape [?]
	 [[{{node audio}}]]
2023-07-04 16:23:48.166730: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs' with dtype float and shape [?]
	 [[{{node inputs}}]]
2023-07-04 16:23:48.189979

INFO:tensorflow:Assets written to: /Users/egyptdj/github/bruit-classification/model/assets


INFO:tensorflow:Assets written to: /Users/egyptdj/github/bruit-classification/model/assets


In [37]:
converter = tf.lite.TFLiteConverter.from_saved_model(MODELDIR)

2023-07-04 16:23:50.596048: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [2]
	 [[{{node inputs_1}}]]
2023-07-04 16:23:50.596100: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [2]
	 [[{{node inputs_1}}]]
2023-07-04 16:23:50.651665: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'inputs_1' with dtype int32 and shape [2]
	 [[{{node inputs_1}}]]
2023-07-04 16:23

In [38]:
tflite_model = converter.convert()

2023-07-04 16:23:52.610271: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'serving_default_audio' with dtype float and shape [?]
	 [[{{node serving_default_audio}}]]
2023-07-04 16:23:52.745162: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2023-07-04 16:23:52.745174: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2023-07-04 16:23:52.745432: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /Users/egyptdj/github/bruit-classification/model
2023-07-04 16:23:52.760120: I tensorflow/cc/saved_model/reader.cc:89] Reading meta graph with tags { serve }
2023-07-04 16:23:52.760133: I tensorflow/cc/saved_model/reader.cc:130] Reading SavedModel debug info (if present) from: /Users/egyptdj/

In [39]:
with open(os.path.join(MODELDIR, 'bruit_yamnet_twostep.tflite'), 'wb') as f:
  f.write(tflite_model)